# 05 — Tabel fitur NSRDB + audit kebocoran (T12)

Satu baris = **(hari target D, jam target 1–24 WIB, period-ending)**. Ramalan diterbitkan **12.00 WIB D-1** (05.00 UTC).

| Kelompok | Fitur | Aturan waktu |
|---|---|---|
| Geometri (deterministik) | `zenith`, `cos_zenith`, `ghi_cs` (Ineichen), `ghi_extra`, `hour_sin/cos`, `doy_sin/cos` | boleh memakai jam target |
| Lag observasi | `kc_same_hour_lag` (+ `same_hour_from_d2`) | D-1 bila jam itu ≤ 12.00 WIB, selain itu D-2 (keputusan #8) |
| | `kc_mean_d1_to_issue`, `kc_mean_d1_morning` (08–12 WIB) | D-1 s.d. penerbitan |
| | `kc_mean_d2` | D-2 penuh |
| | `kc_at_issue`, `*_at_issue` (suhu, RH, titik embun, AOD) | jam yang berakhir 12.00 WIB D-1 |
| Klimatologi | `kc_clim` (bulan × jam) | dari data latih 2016–2018 saja |
| Target & mask (bukan fitur) | `y_kc`, `y_ghi`, `is_day`, `eval_valid` | — |

Kode: `src/features.py`. Tes: `tests/test_leakage.py` (termasuk 2 tes negatif). Fitur GFS ditambahkan di T18.

## 0. Setup

In [8]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import importlib, features as F
importlib.reload(F)

CFG = yaml.safe_load(open(ROOT / "configs" / "config.yaml"))
INTERIM, PROC, RES = ROOT / "data/interim", ROOT / "data/processed", ROOT / "results"
PROC.mkdir(parents=True, exist_ok=True)
print("issue_time_utc:", CFG["forecast"]["issue_time_utc"], "| clear-sky:", CFG["physics"]["clearsky_ref"])

issue_time_utc: 05:00 | clear-sky: ineichen


## 1. Bangun tabel

In [9]:
hourly = pd.read_parquet(INTERIM / "nsrdb_hourly_kc.parquet")
need = ["kc", "ghi", "ghi_cs", "solar_zenith_angle", "interp_frac"]
assert all(c in hourly for c in need), f"Kolom kurang: {[c for c in need if c not in hourly]} — jalankan notebook 04 dulu"

table = F.build_feature_table(hourly, CFG)
FEATS = [c for c in table.columns if not c.startswith(F.FTM_PREFIX) and c not in
         ("target_day", "target_hour_wib", "target_time_utc", "issue_time_utc",
          "y_ghi", "y_kc", "is_day", "interp_frac", "eval_valid", "split")]
print(f"{len(table):,} baris ({table['target_day'].nunique():,} hari × 24) | {len(FEATS)} fitur")
print(FEATS)

43,848 baris (1,827 hari × 24) | 19 fitur
['zenith', 'cos_zenith', 'ghi_cs', 'ghi_extra', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'kc_same_hour_lag', 'same_hour_from_d2', 'kc_mean_d1_to_issue', 'kc_mean_d1_morning', 'kc_mean_d2', 'kc_at_issue', 'temperature_at_issue', 'relative_humidity_at_issue', 'dew_point_at_issue', 'aerosol_optical_depth_at_issue', 'kc_clim']


## 2. Audit kebocoran (T12.6) — harus 0 baris bocor di semua fitur

In [10]:
rep = F.leakage_report(table)
display(rep)
F.assert_no_leakage(table)
print("✓ Tidak ada fitur yang memakai data setelah 12.00 WIB D-1")

,fitur,baris_bocor,maks_selisih_jam
0,kc_same_hour_lag,0,0.0
1,kc_mean_d1_to_issue,0,0.0
2,kc_mean_d1_morning,0,0.0
3,kc_mean_d2,0,-12.0
4,kc_at_issue,0,0.0
5,temperature_at_issue,0,0.0
6,relative_humidity_at_issue,0,0.0
7,dew_point_at_issue,0,0.0
8,aerosol_optical_depth_at_issue,0,0.0


✓ Tidak ada fitur yang memakai data setelah 12.00 WIB D-1


## 3. Ringkasan per split + kelengkapan fitur

In [11]:
summ = table.groupby("split").agg(hari=("target_day", "nunique"), baris=("y_ghi", "size"),
                                  jam_siang=("is_day", "sum"), jam_eval_valid=("eval_valid", "sum"))
display(summ)

day = table["is_day"]
na = table.loc[day, FEATS].isna().mean().mul(100).round(2).rename("% NaN (jam siang)")
display(na[na > 0].to_frame() if (na > 0).any() else "Tidak ada NaN pada jam siang")
print(f"Jam siang yang k_c jam-samanya diambil dari D-2: {table.loc[day, 'same_hour_from_d2'].mean():.1%}")

,hari,baris,jam_siang,jam_eval_valid
split,,,,
test,366,8784,4174,3415
train,1096,26304,12496,9759
val,365,8760,4161,3289


,% NaN (jam siang)
kc_same_hour_lag,0.21
kc_mean_d1_to_issue,0.05
kc_mean_d1_morning,0.05
kc_mean_d2,0.11
kc_at_issue,0.05
temperature_at_issue,0.05
relative_humidity_at_issue,0.05
dew_point_at_issue,0.05
aerosol_optical_depth_at_issue,0.05


Jam siang yang k_c jam-samanya diambil dari D-2: 48.4%


## 4. Cek cepat: hubungan fitur dengan target (data latih, jam evaluasi valid)
Hanya untuk melihat fitur mana yang informatif — **bukan** hasil dan tidak dilaporkan. Untuk day-ahead tanpa NWP, korelasi lag dengan target biasanya lemah–sedang; kalau ada yang sangat tinggi (> 0,8), curigai kebocoran (risiko #5).

In [12]:
tr = table[(table["split"] == "train") & table["eval_valid"]]
corr = tr[FEATS + ["y_kc"]].corr(method="spearman")["y_kc"].drop("y_kc").sort_values(key=abs, ascending=False)
display(corr.round(3).to_frame("Spearman vs y_kc"))
if (corr.drop(["kc_clim"], errors="ignore").abs() > 0.8).any():
    print("⚠ Ada fitur berkorelasi > 0,8 dengan target — periksa kebocoran!")

,Spearman vs y_kc
kc_clim,0.607
kc_same_hour_lag,0.461
hour_sin,0.385
kc_mean_d1_morning,0.377
kc_mean_d1_to_issue,0.367
same_hour_from_d2,-0.362
ghi_extra,-0.335
kc_at_issue,0.332
doy_cos,-0.330
kc_mean_d2,0.311


## 5. Contoh satu hari target — untuk memahami garis waktu

In [13]:
d0 = table.loc[table["split"] == "val", "target_day"].iloc[24 * 150]
ex = table[table["target_day"] == d0].set_index("target_hour_wib")
show = ["target_time_utc", "issue_time_utc", "zenith", "ghi_cs", "kc_same_hour_lag", "same_hour_from_d2",
        "kc_mean_d1_to_issue", "kc_mean_d2", "kc_clim", "y_kc", "eval_valid"]
print(f"Hari target {d0:%d %b %Y}; ramalan diterbitkan {ex['issue_time_utc'].iloc[0]:%d %b %Y %H:%M} UTC (12.00 WIB D-1)")
display(ex.loc[5:19, show].round({c: 3 for c in show if ex[c].dtype.kind == "f"}))

Hari target 31 May 2019; ramalan diterbitkan 30 May 2019 05:00 UTC (12.00 WIB D-1)


,target_time_utc,issue_time_utc,zenith,ghi_cs,kc_same_hour_lag,same_hour_from_d2,kc_mean_d1_to_issue,kc_mean_d2,kc_clim,y_kc,eval_valid
target_hour_wib,,,,,,,,,,,
5,2019-05-30 22:00:00+00:00,2019-05-30 05:00:00+00:00,111.580,0.000,NaN,0,1.074,0.801,NaN,NaN,False
6,2019-05-30 23:00:00+00:00,2019-05-30 05:00:00+00:00,97.673,0.000,NaN,0,1.074,0.801,NaN,NaN,False
7,2019-05-31 00:00:00+00:00,2019-05-30 05:00:00+00:00,83.703,47.788,1.464,0,1.074,0.801,1.358,1.440,True
8,2019-05-31 01:00:00+00:00,2019-05-30 05:00:00+00:00,70.283,258.813,1.070,0,1.074,0.801,0.928,0.958,True
9,2019-05-31 02:00:00+00:00,2019-05-30 05:00:00+00:00,57.260,488.458,1.008,0,1.074,0.801,0.884,0.931,True
10,2019-05-31 03:00:00+00:00,2019-05-30 05:00:00+00:00,45.220,678.752,0.986,0,1.074,0.801,0.871,0.989,True
11,2019-05-31 04:00:00+00:00,2019-05-30 05:00:00+00:00,35.252,810.653,0.972,0,1.074,0.801,0.818,0.982,True
12,2019-05-31 05:00:00+00:00,2019-05-30 05:00:00+00:00,29.535,873.497,0.940,0,1.074,0.801,0.738,0.981,True
13,2019-05-31 06:00:00+00:00,2019-05-30 05:00:00+00:00,30.463,862.497,0.771,1,1.074,0.801,0.686,0.977,True


## 6. Simpan + dokumentasi fitur (bahan Bab III, T19.6)

In [14]:
table.to_parquet(PROC / "features_nsrdb.parquet")

rules = {
    "zenith": ("geometri", "deterministik (jam target)"), "cos_zenith": ("geometri", "deterministik (jam target)"),
    "ghi_cs": ("geometri", "deterministik, Ineichen + Linke klimatologi"),
    "ghi_extra": ("geometri", "deterministik"), "hour_sin": ("geometri", "deterministik"),
    "hour_cos": ("geometri", "deterministik"), "doy_sin": ("geometri", "deterministik"), "doy_cos": ("geometri", "deterministik"),
    "kc_same_hour_lag": ("lag observasi", "jam sama D-1 bila ≤ 12.00 WIB D-1, selain itu D-2"),
    "same_hour_from_d2": ("lag observasi", "penanda sumber kc_same_hour_lag"),
    "kc_mean_d1_to_issue": ("lag observasi", "rata-rata k_c D-1 jam 01–12 WIB"),
    "kc_mean_d1_morning": ("lag observasi", "rata-rata k_c D-1 jam 08–12 WIB"),
    "kc_mean_d2": ("lag observasi", "rata-rata k_c D-2 penuh"),
    "kc_at_issue": ("lag observasi", "jam yang berakhir 12.00 WIB D-1"),
    "kc_clim": ("klimatologi", "rata-rata k_c bulan × jam, data latih 2016–2018"),
}
doc = pd.DataFrame([{"fitur": f, "kelompok": rules.get(f, ("lag observasi", "jam yang berakhir 12.00 WIB D-1"))[0],
                     "aturan_waktu": rules.get(f, ("", "jam yang berakhir 12.00 WIB D-1"))[1]} for f in FEATS])
doc.to_csv(RES / "feature_list_nsrdb.csv", index=False)
display(doc)
print(f"""
Salin ke decisions.md:
- {pd.Timestamp.today():%Y-%m-%d} | T12: tabel (hari target D, jam 1–24 WIB period-ending), {len(table):,} baris, {len(FEATS)} fitur NSRDB. Penerbitan 12.00 WIB D-1 (asumsi kerja, menunggu konfirmasi Pemb. II). Keputusan #8 diterapkan: k_c jam sama dari D-1 bila ≤ penerbitan, selain itu D-2 ({table.loc[table['is_day'], 'same_hour_from_d2'].mean():.0%} jam siang dari D-2); rata-rata D-1 s.d. penerbitan; D-2 penuh. Audit kebocoran: 0 baris bocor; tes negatif lulus. Latensi data satelit diasumsikan nol. | results/feature_list_nsrdb.csv | Bab III""")

,fitur,kelompok,aturan_waktu
0,zenith,geometri,deterministik (jam target)
1,cos_zenith,geometri,deterministik (jam target)
2,ghi_cs,geometri,"deterministik, Ineichen + Linke klimatologi"
3,ghi_extra,geometri,deterministik
4,hour_sin,geometri,deterministik
5,hour_cos,geometri,deterministik
6,doy_sin,geometri,deterministik
7,doy_cos,geometri,deterministik
8,kc_same_hour_lag,lag observasi,"jam sama D-1 bila ≤ 12.00 WIB D-1, selain itu D-2"
9,same_hour_from_d2,lag observasi,penanda sumber kc_same_hour_lag



Salin ke decisions.md:
- 2026-11-27 | T12: tabel (hari target D, jam 1–24 WIB period-ending), 43,848 baris, 19 fitur NSRDB. Penerbitan 12.00 WIB D-1 (asumsi kerja, menunggu konfirmasi Pemb. II). Keputusan #8 diterapkan: k_c jam sama dari D-1 bila ≤ penerbitan, selain itu D-2 (48% jam siang dari D-2); rata-rata D-1 s.d. penerbitan; D-2 penuh. Audit kebocoran: 0 baris bocor; tes negatif lulus. Latensi data satelit diasumsikan nol. | results/feature_list_nsrdb.csv | Bab III
